# Exercise 1 (Association Rules) 

In [93]:
def calculate_support(items, transactions): 
    """
    probability - how likely this film occurs in one transaction 
    returns Number_of_occurences/all_transactions
    """
    occurence = 0
    for tra in transactions: 
        all_ocuring = True 
        for item in items: 
            if item not in tra: 
                all_ocuring=False 
                break 
        if all_ocuring: 
            occurence +=1 
    
    return occurence/len(transactions)


def get_all_items(transactions): 
    """returns all items from the transactions"""
    itemlist = []
    for t in transactions: 
        for e in t: 
            if e not in itemlist: 
                itemlist.append(e)
    return itemlist

In [94]:
def calculate_L1(transactions, minsup): 
    """returns all itemsets with len=1 and support > minsup"""
    items = get_all_items(transactions)

    L1 = []
    for item in items: 
        supp = calculate_support([item], transactions)
        if minsup <= supp: 
            L1.append([[item],supp])

    return sorted(L1, key=lambda x: x[0])


In [95]:
def apriori_gen(L_k_1): 
    """
    generates k+1 sets from k sets 
    combine two k sets if k-1 items match 
    """

    C_k = [] 
    L_list = []
    for item in L_k_1: 
        L_list.append(item[0])

    for i in range(0,len(L_list)): 
        for j in range(i+1, len(L_list)): 

            set1 = sorted(L_list[i])
            set2 = sorted(L_list[j])
            if set1[:-1] == set2[:-1]: 
                candidate = sorted (set(set1) | set(set2) )
                C_k.append(candidate)

    C_k = [list(x) for x in set(tuple(sorted(c)) for c in C_k)] # remove duplikates 
    return C_k 
    

In [96]:
def calculate_confidence(left, right, transactions): 
    """
    essentially P(B|A) = P(A u B)/P(A)
    we have A (left) how likely is B (right)
    """
    combined_supp = calculate_support(left+right, transactions)
    left_supp = calculate_support(left, transactions)

    if left_supp == 0:
        return 0
    
    confidence = combined_supp/ left_supp
    return confidence

In [97]:
##########################################
# I -> all possibles items 
# D -> transactions 
# L_i 
# C_i 

def apriori(transactions, minsupp): 
    # Calculate items 
    items = get_all_items(transactions)
    print(f"Found items: {items}")
    print(f"Minsupp: {minsupp} ({minsupp*100}%)")


    L1 = calculate_L1(transactions, minsupp)
    print("L1 set")
    print(L1)

    L_all = L1.copy()
    L_k_1 = L1.copy() 
    k = 2
    print("\n")
    while L_k_1: 
        print("##########################\n======= Iteration: ", k)
        C_k = apriori_gen(L_k_1)

        if not C_k: 
            print("Didnt found any new C_k")
            break 
        print("C_", k, " : ", C_k)


        L_k = []

        for elem in C_k: 
            supp = calculate_support(elem, transactions)
            if supp >= minsupp: 
                L_k.append([elem,supp])

        if not L_k: 
            print("Didnt found any L_k")
            break 

        print("L_", k, " set")
        print(L_k)
        L_k_1 = L_k 
        L_all.extend(L_k)
        k+=1 
        print("\n")
    print("\n")
    print("Combined sets: ")
    print(L_all)
    return L_all 


In [112]:
def generate_rules(L_all, transactions, minconf): 

    rules = []



    for items, supp in L_all: 
        # if we have only 1 item there is nothing else to recommend
        if len(items)< 2 :  
            continue 

        print("Calc rules from set: ", items)

        for i, resulting_item in enumerate(items): 
            left = items[:i] + items[i+1:] 
            confidence = calculate_confidence(left, [resulting_item], transactions)

            rules.append({
                'left': left,
                'consequence': resulting_item,
                'support': supp,
                'confidence': confidence,
                'valid': confidence >= minconf
            })
    return rules 



def get_recommendations(rules, user_items,): 

    print("User listened to : ", user_items)

    recommendations = set() 
    for rule in rules: 
        if not rule["valid"]: 
            continue 
        if set(rule['left']) == set(user_items):
            if rule["consequence"] not in user_items: 
                recommendations.add(rule["consequence"])

    if recommendations:
        print(f"\nRecommendations: {sorted(list(recommendations))}")
    else:
        print("No recommendations found")
    
    return recommendations


In [116]:
# Data 
transactions = [
    ['Highway to Hell', 'Country Roads', 'Learn to Fly', 'Dancing Queen'],
    ['Country Roads', 'Learn to Fly'],
    ['Highway to Hell', 'My Heart Will Go On', 'Dancing Queen'],
    ['Dancing Queen', 'Learn to Fly'],
    ['Highway to Hell', 'Learn to Fly', 'Dancing Queen', 'My Heart Will Go On'],
    ['Highway to Hell', 'Learn to Fly']
]

target_transaction = ['Learn to Fly']




print("\n" + "="*60)
print("APRIORI ALGORITHM - MUSIC RECOMMENDATIONS")
print("="*60 + "\n")

minsupp = 0.5
all_itemsets = apriori(transactions, minsupp)

print("\n" + "="*60)
print("Common Sets")
print("="*60)
print(f"\nSets (Support >= {minsupp:.0%}):\n")
for s in all_itemsets:
    print(s)




print("\n" + "="*60)
print("Rules")
print("\n" + "="*60)
minconf = 0.6  
rules = generate_rules(all_itemsets, transactions, minconf)
for rule in rules: 
    print(rule)

print("\n" + "="*60)
print("Recommendation")
print("\n" + "="*60)
recommendations = get_recommendations(rules, target_transaction)


APRIORI ALGORITHM - MUSIC RECOMMENDATIONS

Found items: ['Highway to Hell', 'Country Roads', 'Learn to Fly', 'Dancing Queen', 'My Heart Will Go On']
Minsupp: 0.5 (50.0%)
L1 set
[[['Dancing Queen'], 0.6666666666666666], [['Highway to Hell'], 0.6666666666666666], [['Learn to Fly'], 0.8333333333333334]]


##########################
======= Iteration:  2
C_ 2  :  [['Highway to Hell', 'Learn to Fly'], ['Dancing Queen', 'Highway to Hell'], ['Dancing Queen', 'Learn to Fly']]
L_ 2  set
[[['Highway to Hell', 'Learn to Fly'], 0.5], [['Dancing Queen', 'Highway to Hell'], 0.5], [['Dancing Queen', 'Learn to Fly'], 0.5]]


##########################
======= Iteration:  3
C_ 3  :  [['Dancing Queen', 'Highway to Hell', 'Learn to Fly']]
Didnt found any L_k


Combined sets: 
[[['Dancing Queen'], 0.6666666666666666], [['Highway to Hell'], 0.6666666666666666], [['Learn to Fly'], 0.8333333333333334], [['Highway to Hell', 'Learn to Fly'], 0.5], [['Dancing Queen', 'Highway to Hell'], 0.5], [['Dancing Queen'

# Exercise 2 (Association Rules in Action)

In [117]:

import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

pd.set_option('display.max_colwidth', None)

#################################################################################
#  Load Data
ratings = pd.read_csv("./ml-latest-small/ratings.csv")
movies = pd.read_csv("./ml-latest-small/movies.csv")

df = pd.merge(ratings, movies, on="movieId")



print(f"Anzahl Filme nach Filter: {df['movieId'].nunique()}")

#################################################################################
#  User-Item-Matrix erstellen
user_movie_matrix = df.pivot_table(
    index="userId",
    columns="title",
    values="rating",
    aggfunc=lambda x: 1,
    fill_value=0
)

user_movie_matrix = user_movie_matrix.astype(bool)

print(f"Matrix Shape: {user_movie_matrix.shape}")

#################################################################################
#  Association rules 
frequent_itemsets = fpgrowth(
    user_movie_matrix,
    min_support=0.1,   
    use_colnames=True,
    max_len=2    # only two combinations reduce possibilieties and compute power        
)

print("Frequent Itemsets:")
print(frequent_itemsets.head())


rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)





rules = rules.sort_values(by="confidence", ascending=False)

print("\nTop 10 Regeln:")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))







Anzahl Filme nach Filter: 9724
Matrix Shape: (610, 9719)
Frequent Itemsets:
    support                                     itemsets
0  0.539344                        (Forrest Gump (1994))
1  0.503279                        (Pulp Fiction (1994))
2  0.457377           (Silence of the Lambs, The (1991))
3  0.455738                         (Matrix, The (1999))
4  0.411475  (Star Wars: Episode IV - A New Hope (1977))

Top 10 Regeln:
                                                antecedents  \
8717                       (Godfather: Part II, The (1974))   
5887                                  (Philadelphia (1993))   
9133    (Pirates of the Caribbean: Dead Man's Chest (2006))   
1023                                (Mrs. Doubtfire (1993))   
7122                          (Matrix Reloaded, The (2003))   
8059                             (Kill Bill: Vol. 2 (2004))   
7411  (Star Wars: Episode III - Revenge of the Sith (2005))   
5116                           (In the Line of Fire (1993))   

# 2b) qualitative study

In [118]:
def recommend(movie_name, rules_df, top_n=5):
    recs = rules_df[rules_df['antecedents'].apply(lambda x: movie_name in x)]
    recs = recs.sort_values(by="confidence", ascending=False)
    
    return recs[['consequents', 'confidence', 'lift']].head(top_n)

In [125]:
#textsearch 
df[df["title"].str.contains(r"toy story", case=False, regex=True, na=False)]

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
516,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
874,7,1,4.5,1106635946,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
927,7,3114,4.5,1106635500,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
1434,15,1,2.5,1510577970,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
...,...,...,...,...,...,...
99121,608,3114,2.5,1117504405,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
99497,609,1,3.0,847221025,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
99534,610,1,5.0,1479542900,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
99709,610,3114,5.0,1479542923,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy


In [126]:
seed_movie = "Star Wars: Episode IV - A New Hope (1977)"

print(f"\nRecommendation for: {seed_movie}")
print(recommend(seed_movie, rules))



Recommendation for: Star Wars: Episode IV - A New Hope (1977)
                                                                          consequents  \
114                           (Star Wars: Episode V - The Empire Strikes Back (1980))   
19                                                               (Matrix, The (1999))   
291                               (Star Wars: Episode VI - Return of the Jedi (1983))   
20                                                              (Forrest Gump (1994))   
255  (Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981))   

     confidence      lift  
114    0.756972  2.188403  
19     0.729084  1.599788  
291    0.697211  2.169892  
20     0.653386  1.211446  
255    0.633466  1.932072  


In this case the recommendations make sense. 
Star wars 5 and 6 are obvious recommendations. I'm a bit confused that it doesn't recommend anything from before (e.g. I or II). 
Because thinking about the algorithm if most people watch 1 -> 6 the algorithm should favor recommending earlier films than later films. 
But maybe this is because 4-6 was first released and therefore the batch 4-6 is watched more together than 1-4. 

Matrix and indiana jones also make sense because I feel like they have a similar vibe. I don't know about forrest gump, if that fits. 

In [127]:
seed_movie = "Star Wars: Episode V - The Empire Strikes Back (1980)"

print(f"Recommendation for: {seed_movie}")
print(recommend(seed_movie, rules))

Recommendation for: Star Wars: Episode V - The Empire Strikes Back (1980)
                                                                          consequents  \
115                                       (Star Wars: Episode IV - A New Hope (1977))   
117                                                              (Matrix, The (1999))   
293                               (Star Wars: Episode VI - Return of the Jedi (1983))   
118                                                             (Forrest Gump (1994))   
259  (Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981))   

     confidence      lift  
115    0.900474  2.188403  
117    0.819905  1.799073  
293    0.767773  2.389496  
118    0.701422  1.300509  
259    0.701422  2.139336  


Here we can see that as earlier expected, the algorithm really likes to recommend the earlier episodes rather than the ones coming after. 

In [128]:
seed_movie = "Star Wars: Episode III - Revenge of the Sith (2005)"
print(f"Recommendation for: {seed_movie}")
print(recommend(seed_movie, rules))

Recommendation for: Star Wars: Episode III - Revenge of the Sith (2005)
                                                  consequents  confidence  \
7411              (Star Wars: Episode IV - A New Hope (1977))    0.935897   
7412  (Star Wars: Episode V - The Empire Strikes Back (1980))    0.923077   
7427                                     (Matrix, The (1999))    0.923077   
7415          (Lord of the Rings: The Two Towers, The (2002))    0.897436   
7416      (Star Wars: Episode VI - Return of the Jedi (1983))    0.884615   

          lift  
7411  2.274492  
7412  2.668611  
7427  2.025457  
7415  2.911893  
7416  2.753140  


And here it also recommends movies from the different Star Wars era (4-6). Interestingly the algorithm doesn't recommend the movies 1-2. 

But in general I can say that the recommendations make sense. If I watched one of the Star Wars movies I would be interested in the other Star Wars movies, as well as the other "random" movies like Matrix or lord of the rings. 

In [129]:
seed_movie = "Toy Story (1995)"
print(f"Recommendation for: {seed_movie}")
print(recommend(seed_movie, rules))

Recommendation for: Toy Story (1995)
                                     consequents  confidence      lift
92                         (Forrest Gump (1994))    0.716279  1.328055
94                         (Pulp Fiction (1994))    0.655814  1.303083
110           (Shawshank Redemption, The (1994))    0.637209  1.226176
96   (Star Wars: Episode IV - A New Hope (1977))    0.623256  1.514685
98                        (Jurassic Park (1993))    0.613953  1.573578


In [130]:
seed_movie = "Jungle Book 2, The (2003)"
print(f"Recommendation for: {seed_movie}")
print(recommend(seed_movie, rules))

Recommendation for: Jungle Book 2, The (2003)
Empty DataFrame
Columns: [consequents, confidence, lift]
Index: []


Im not so happy with those recommendations. The recommendations don't really match what I would show to kids who have watched Toy story. 
The problem could be that many kids use the accounts from their parents. Hence adult films are matched with kid films. 
Also I could imagine that lesser known films (e.g Star wars -> 2000 ratings, some old Jungle Book film -> 60) have bigger problems with generating recommendations. For the jungle book it couldnt give me a single recommendation. For that I probably would have to lower the thresholds. 